In [1]:
import pandas as pd 
import datetime as dt
import matplotlib.pyplot as plt
import yfinance as yf
import numpy as np
from scipy.stats import norm

In [2]:
years  =   15
end_date = dt.datetime.now()
start_date = end_date - dt.timedelta(days=years * 365)

In [3]:
tickers =['SPY', 'BND', 'GLD', 'QQQ', 'VTI']

In [4]:
adj_close = pd.DataFrame()
for ticker in tickers:
    data= yf.download(ticker, start=start_date, end=end_date, auto_adjust=False)
    adj_close[ticker] = data['Adj Close']

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [5]:
print(adj_close.head())

                  SPY        BND         GLD        QQQ        VTI
Date                                                              
2010-07-28  84.373116  53.109604  113.779999  40.316116  43.211437
2010-07-29  83.962036  53.168137  114.290001  40.018440  42.989552
2010-07-30  83.946785  53.246185  115.489998  40.105988  43.035469
2010-08-02  85.842400  53.234447  115.540001  40.858917  43.961197
2010-08-03  85.431297  53.377960  115.989998  40.683819  43.724041


In [6]:
log_returns = np.log(adj_close / adj_close.shift(1)).dropna()

In [7]:
portfolio_value = 1000000  # Initial portfolio value
weights = np.array([1/len(tickers)] * len(tickers))  # Equal weights for simplicity
print("Weights:", weights)

Weights: [0.2 0.2 0.2 0.2 0.2]


In [8]:
historical_returns = (log_returns * weights).sum(axis=1)
print("Historical Returns:", historical_returns.head())

Historical Returns: Date
2010-07-29   -0.002374
2010-07-30    0.002997
2010-08-02    0.012485
2010-08-03   -0.001585
2010-08-04    0.005777
dtype: float64


In [9]:
days = 5
historical_x_day_returns = historical_returns.rolling(window=days).sum().dropna()

In [10]:
cov_matrix = log_returns.cov() * 252  # Annualize the covariance matrix
portfolio_std_dev =  np.sqrt(weights.T @ cov_matrix @ weights)  # Portfolio standard deviation

In [11]:
confidence_intervals = [0.9, 0.95, 0.99]

VaRs= []

for cl in confidence_intervals:
    z_score = norm.ppf(cl)
    VaR= portfolio_value * portfolio_std_dev * z_score * np.sqrt(days/252)  # Annualize the VaR
    VaRs.append(VaR)

In [12]:
print(f'{"Confidence Intervals":<20} {"Value at Risk":<20}')
print("-"*40)

for cl, var in zip(confidence_intervals, VaRs):
    print(f'{cl * 100:>6.0f}%: {"":<12} ${var:>10,.2f}')

Confidence Intervals Value at Risk       
----------------------------------------
    90%:              $ 21,069.49
    95%:              $ 27,042.39
    99%:              $ 38,246.57
